## IMPORTING DEPENDENCIES

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
import pandas as pd
from sqlalchemy import create_engine


In [2]:
# Initialising SparkSession
spark = SparkSession.builder.appName('NugaBankETL').getOrCreate()


In [3]:
spark

### EXTRACTION

In [4]:
# Reading csv file 
nuga_bank_df = spark.read.csv(r'DataSet\nuga_bank_transactions.csv', header = True, inferSchema= True)


In [5]:
# Display DataFrame
nuga_bank_df.show(5)

+--------------------+------+----------------+--------------+--------------------+------------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+------------------+--------------------+-------------+-------------+--------+-----+---------+--------------------+--------------------+------+--------------+
|    Transaction_Date|Amount|Transaction_Type| Customer_Name|    Customer_Address|     Customer_City|Customer_State|    Customer_Country|             Company|           Job_Title|               Email|       Phone_Number|Credit_Card_Number|                IBAN|Currency_Code|Random_Number|Category|Group|Is_Active|        Last_Updated|         Description|Gender|Marital_Status|
+--------------------+------+----------------+--------------+--------------------+------------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+------------------+-----

In [6]:
# Show the info/Schema of the data frame
nuga_bank_df.printSchema()

root
 |-- Transaction_Date: timestamp (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Address: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_State: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone_Number: string (nullable = true)
 |-- Credit_Card_Number: long (nullable = true)
 |-- IBAN: string (nullable = true)
 |-- Currency_Code: string (nullable = true)
 |-- Random_Number: double (nullable = true)
 |-- Category: string (nullable = true)
 |-- Group: string (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)
 |-- Description: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital_Status: string (nullable = true)

In [7]:
# Checking for number of rows
num_rows = nuga_bank_df.count()

num_rows

1000000

In [8]:
# Checking for number of clumns
num_columns = len(nuga_bank_df.columns)
num_columns

23

In [9]:
# Checking for Null values
for column in nuga_bank_df.columns:
    print(column, 'Nulls',nuga_bank_df.filter(nuga_bank_df[column].isNull()).count())


Transaction_Date Nulls 0
Amount Nulls 0
Transaction_Type Nulls 0
Customer_Name Nulls 100425
Customer_Address Nulls 100087
Customer_City Nulls 100034
Customer_State Nulls 100009
Customer_Country Nulls 100672
Company Nulls 100295
Job_Title Nulls 99924
Email Nulls 100043
Phone_Number Nulls 100524
Credit_Card_Number Nulls 100085
IBAN Nulls 100300
Currency_Code Nulls 99342
Random_Number Nulls 99913
Category Nulls 100332
Group Nulls 100209
Is_Active Nulls 100259
Last_Updated Nulls 100321
Description Nulls 100403
Gender Nulls 99767
Marital_Status Nulls 99904


In [39]:
# Filling missing/null values
nuga_bank_df_clean = nuga_bank_df.fillna({
    'Customer_Name' : 'Unknown',
    'Customer_Address' : 'Unknown',
    'Customer_City' : 'Unknown',
    'Customer_State' : 'Unknown',
    'Customer_Country' : 'Unknown',
    'Company' : 'Unknown',
    'Job_Title' : 'Unknown',
    'Email' : 'Unknown',
    'Phone_Number' : 'Unknown',
    'Credit_Card_Number' : 0,
    'IBAN' : 'Unknown',
    'Currency_Code' : 'Unknown',
    'Random_Number' : 0.0,
    'Category' : 'Unknown',
    'Group' : 'Unknown',
    'Is_Active' : 'Unknown',
    'Description' : 'Unknown',
    'Gender' : 'Unknown',
    'Marital_Status' : 'Unknown'
})

In [40]:
# Re-checking for Null values
for column in nuga_bank_df_clean.columns:
    print(column, 'Nulls',nuga_bank_df_clean.filter(nuga_bank_df_clean[column].isNull()).count())


Transaction_Date Nulls 0
Amount Nulls 0
Transaction_Type Nulls 0
Customer_Name Nulls 0
Customer_Address Nulls 0
Customer_City Nulls 0
Customer_State Nulls 0
Customer_Country Nulls 0
Company Nulls 0
Job_Title Nulls 0
Email Nulls 0
Phone_Number Nulls 0
Credit_Card_Number Nulls 0
IBAN Nulls 0
Currency_Code Nulls 0
Random_Number Nulls 0
Category Nulls 0
Group Nulls 0
Is_Active Nulls 0
Last_Updated Nulls 100321
Description Nulls 0
Gender Nulls 0
Marital_Status Nulls 0


In [42]:
# Dropping null values from the last updated column 
nuga_bank_df_clean = nuga_bank_df_clean.na.drop(subset=['Last_Updated'])

In [43]:
# Re-checking for Null values
for column in nuga_bank_df_clean.columns:
    print(column, 'Nulls',nuga_bank_df_clean.filter(nuga_bank_df_clean[column].isNull()).count())

Transaction_Date Nulls 0
Amount Nulls 0
Transaction_Type Nulls 0
Customer_Name Nulls 0
Customer_Address Nulls 0
Customer_City Nulls 0
Customer_State Nulls 0
Customer_Country Nulls 0
Company Nulls 0
Job_Title Nulls 0
Email Nulls 0
Phone_Number Nulls 0
Credit_Card_Number Nulls 0
IBAN Nulls 0
Currency_Code Nulls 0
Random_Number Nulls 0
Category Nulls 0
Group Nulls 0
Is_Active Nulls 0
Last_Updated Nulls 0
Description Nulls 0
Gender Nulls 0
Marital_Status Nulls 0


In [14]:
# Checking for number of rows
num_rows = nuga_bank_df_clean.count()

num_rows

899679

In [44]:
# Testing for aggregated summary where applicable by using the describe function
nuga_bank_df_clean.describe().show()

+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+-----------------+--------+-------+---------+--------------------+-------+--------------+
|summary|            Amount|Transaction_Type|Customer_Name|    Customer_Address|Customer_City|Customer_State|Customer_Country|      Company|         Job_Title|              Email|        Phone_Number|  Credit_Card_Number|                IBAN|Currency_Code|    Random_Number|Category|  Group|Is_Active|         Description| Gender|Marital_Status|
+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+-----------------+--------+-------+---------+-------

In [45]:
# Display columns of cleaned DataFrame
nuga_bank_df_clean.columns

['Transaction_Date',
 'Amount',
 'Transaction_Type',
 'Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Company',
 'Job_Title',
 'Email',
 'Phone_Number',
 'Credit_Card_Number',
 'IBAN',
 'Currency_Code',
 'Random_Number',
 'Category',
 'Group',
 'Is_Active',
 'Last_Updated',
 'Description',
 'Gender',
 'Marital_Status']

In [ ]:
# creating the transaction table
transactions = nuga_bank_df_clean.select('Transaction_Date','Amount','Transaction_Type')

In [19]:
transactions.show()

+--------------------+------+----------------+
|    Transaction_Date|Amount|Transaction_Type|
+--------------------+------+----------------+
|2024-03-23 15:38:...| 34.76|      Withdrawal|
|2024-04-22 19:15:...|163.92|      Withdrawal|
|2024-04-12 19:46:...|386.32|      Withdrawal|
|2024-04-17 15:29:...|407.15|         Deposit|
|2024-02-10 01:51:...|161.31|         Deposit|
|2024-02-10 22:56:...|764.34|        Transfer|
|2024-04-07 00:07:...|734.59|         Deposit|
|2024-03-08 01:51:...|592.43|         Deposit|
|2024-02-01 12:34:...| 927.1|         Deposit|
|2024-03-22 16:46:...| 66.59|        Transfer|
|2024-04-23 13:30:...| 246.3|      Withdrawal|
|2024-01-13 01:22:...|782.32|      Withdrawal|
|2024-02-25 15:16:...|818.42|      Withdrawal|
|2024-01-01 20:55:...|352.23|      Withdrawal|
|2024-01-19 00:01:...|316.19|      Withdrawal|
|2024-04-09 14:40:...|662.26|      Withdrawal|
|2024-04-15 04:58:...|893.73|         Deposit|
|2024-04-12 14:32:...|746.22|      Withdrawal|
|2024-02-26 1

In [20]:
#creating transaction_id column
transactions = transactions.withColumn('Transaction_ID', monotonically_increasing_id())

In [21]:
transactions.show()

+--------------------+------+----------------+--------------+
|    Transaction_Date|Amount|Transaction_Type|Transaction_ID|
+--------------------+------+----------------+--------------+
|2024-03-23 15:38:...| 34.76|      Withdrawal|             0|
|2024-04-22 19:15:...|163.92|      Withdrawal|             1|
|2024-04-12 19:46:...|386.32|      Withdrawal|             2|
|2024-04-17 15:29:...|407.15|         Deposit|             3|
|2024-02-10 01:51:...|161.31|         Deposit|             4|
|2024-02-10 22:56:...|764.34|        Transfer|             5|
|2024-04-07 00:07:...|734.59|         Deposit|             6|
|2024-03-08 01:51:...|592.43|         Deposit|             7|
|2024-02-01 12:34:...| 927.1|         Deposit|             8|
|2024-03-22 16:46:...| 66.59|        Transfer|             9|
|2024-04-23 13:30:...| 246.3|      Withdrawal|            10|
|2024-01-13 01:22:...|782.32|      Withdrawal|            11|
|2024-02-25 15:16:...|818.42|      Withdrawal|            12|
|2024-01

In [22]:
#rearranging columns
transactions = transactions.select('Transaction_ID', 'Transaction_Date','Amount','Transaction_Type')


In [23]:
transactions.show()


+--------------+--------------------+------+----------------+
|Transaction_ID|    Transaction_Date|Amount|Transaction_Type|
+--------------+--------------------+------+----------------+
|             0|2024-03-23 15:38:...| 34.76|      Withdrawal|
|             1|2024-04-22 19:15:...|163.92|      Withdrawal|
|             2|2024-04-12 19:46:...|386.32|      Withdrawal|
|             3|2024-04-17 15:29:...|407.15|         Deposit|
|             4|2024-02-10 01:51:...|161.31|         Deposit|
|             5|2024-02-10 22:56:...|764.34|        Transfer|
|             6|2024-04-07 00:07:...|734.59|         Deposit|
|             7|2024-03-08 01:51:...|592.43|         Deposit|
|             8|2024-02-01 12:34:...| 927.1|         Deposit|
|             9|2024-03-22 16:46:...| 66.59|        Transfer|
|            10|2024-04-23 13:30:...| 246.3|      Withdrawal|
|            11|2024-01-13 01:22:...|782.32|      Withdrawal|
|            12|2024-02-25 15:16:...|818.42|      Withdrawal|
|       

In [24]:
# creating the customer table
customers = nuga_bank_df_clean.select('Customer_Name', 'Customer_Address', 'Customer_City', 'Customer_State', 'Customer_Country','Email',
 'Phone_Number').distinct()
# Creating the customer_id column
customers = customers.withColumn('Customer_ID', monotonically_increasing_id())
# Rearrange the columns
customers = customers.select('Customer_ID', 'Customer_Name', 'Customer_Address', 'Customer_City', 'Customer_State', 'Customer_Country','Email',
 'Phone_Number')

In [25]:
customers.show()

+-----------+------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+
|Customer_ID|     Customer_Name|    Customer_Address|       Customer_City|Customer_State|    Customer_Country|               Email|        Phone_Number|
+-----------+------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+
|          0|    Miguel Leonard|262 Beck Expressw...|             Unknown| West Virginia|             Eritrea| zweaver@example.net|             Unknown|
|          1|           Unknown|             Unknown|         Evanchester|        Oregon|             Uruguay|             Unknown| (384)778-9942x91236|
|          2|    Michael Murphy|894 Williams Ridg...|       Dominguezview|      New York|              Sweden|kristinstanley@ex...|+1-693-739-2204x8851|
|          3|    Tina Gutierrez|    415 Taylor Knoll|           Donnastad|South Ca

In [26]:
#creating the employee table
employee = nuga_bank_df_clean.select('Company', 'Job_Title', 'Gender', 'Marital_Status').distinct() 
# Creating the customer_id column
employee = employee.withColumn('Employee_ID', monotonically_increasing_id())
# Rearrange the columns
employee = employee.select('Employee_ID', 'Company', 'Job_Title', 'Gender', 'Marital_Status')


In [27]:
employee.show()


+-----------+--------------------+--------------------+-------+--------------+
|Employee_ID|             Company|           Job_Title| Gender|Marital_Status|
+-----------+--------------------+--------------------+-------+--------------+
|          0|         Price Group|             Unknown|   Male|        Single|
|          1|Rhodes, King and ...| Trade mark attorney|   Male|       Unknown|
|          2|Schmidt, Morgan a...|     Engineer, water| Female|        Single|
|          3|       Johnson Group|  Forensic scientist|   Male|       Unknown|
|          4|     Phillips-Prince|Production assist...|Unknown|        Single|
|          5|      Henry and Sons|Engineer, civil (...| Female|       Married|
|          6|Thompson, Johnson...|Exercise physiolo...|  Other|       Unknown|
|          7|Hernandez, Johnso...|Forensic psycholo...|Unknown|      Divorced|
|          8|Carrillo, Schwart...| Solicitor, Scotland| Female|        Single|
|          9|         Olson-Lucas| Magazine journali

In [28]:
# creating the facts table
facts_table = nuga_bank_df_clean.join(customers,['Customer_Name', 'Customer_Address', 'Customer_City', 'Customer_State', \
                                                 'Customer_Country','Email','Phone_Number'], 'left') \
                                .join(transactions,['Transaction_Date','Amount','Transaction_Type'], 'left') \
                                .join(employee,['Company', 'Job_Title', 'Gender', 'Marital_Status'], 'left') \
                                .select('Customer_ID', 'Transaction_ID', 'Employee_ID', 'Credit_Card_Number','IBAN', \
                                        'Currency_Code','Random_Number','Category','Group','Is_Active','Last_Updated','Description')

In [29]:
facts_table.show()

+------------+--------------+-----------+-------------------+--------------------+-------------+-------------+--------+-------+---------+--------------------+--------------------+
| Customer_ID|Transaction_ID|Employee_ID| Credit_Card_Number|                IBAN|Currency_Code|Random_Number|Category|  Group|Is_Active|        Last_Updated|         Description|
+------------+--------------+-----------+-------------------+--------------------+-------------+-------------+--------+-------+---------+--------------------+--------------------+
|111669155410|    8589934611|       5805|   2395250517931689|GB59IBBF016381524...|          GEL|       7726.0|       B|      Z|       No|2021-10-31 16:19:...|Rule study meet c...|
|103079233043|   42949672969|      18106|                  0|GB07QTMD592344533...|      Unknown|       3854.0|       A|      Y|      Yes|2021-10-24 10:14:...|Me whose himself ...|
| 68719508958|   77309411334|      28519|   3509830057102204|GB98OPAD097093131...|          MRO|    

In [30]:
# saving the transformed data to parquet file
transactions.write.mode('overwrite').parquet(r'DataSet/transactions')
customers.write.mode('overwrite').parquet(r'DataSet/customers')
employee.write.mode('overwrite').parquet(r'DataSet/employee')
facts_table.write.mode('overwrite').parquet(r'DataSet/facts_table')

In [31]:
# saving the transformed data to csv file
transactions.write.mode('overwrite').option('header', True).csv(r'DataSet/transformedData/csv/transactions')
customers.write.mode('overwrite').option('header', True).csv(r'DataSet/transformedData/csv/customers')
employee.write.mode('overwrite').option('header', True).csv(r'DataSet/transformedData/csv/employee')
facts_table.write.mode('overwrite').option('header', True).csv(r'DataSet/transformedData/csv/facts_table')


### LOADING PHASE

In [32]:
# Convert spark df to pandas df
transactions_pd_df = transactions.toPandas()
customers_pd_df = customers.toPandas()
employee_pd_df = employee.toPandas()
facts_table_pd_df = facts_table.toPandas()


In [47]:
# importing some dependencies
from dotenv import load_dotenv
import os

In [48]:
# loading dotenv and defining connection parameters
load_dotenv()

DB_USER = os.getenv('db_user')
DB_PASSWORD = os.getenv('db_password')
DB_HOST = os.getenv('db_host')
DB_PORT = os.getenv('db_port')
DB_NAME = os.getenv('db_name')

In [49]:
# Creating the connection url
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Creating the connection engine using the url
engine = create_engine(db_url)

# Loading the tables to sql
transactions_pd_df.to_sql('Transactions', engine, if_exists='replace', index= False)
customers_pd_df.to_sql('Customers', engine, if_exists='replace', index= False)
employee_pd_df.to_sql('Employees', engine, if_exists='replace', index= False)
facts_table_pd_df.to_sql('Facts_Table', engine, if_exists='replace', index= False)

print('Data loaded into database successfully')



Data loaded into database successfully
